In [23]:
from pathlib import Path
import runpy
import pandas as pd

RAIZ = Path("/home/saurin/sistema-servicios-ia")

catalogo = runpy.run_path(
    str(RAIZ / "ml/training/catalogo.py")
)

filas = []

for categoria, informacion in catalogo["FALLAS"].items():
    for numero, texto in enumerate(informacion["frases"], start=1):
        filas.append({
            "familia": f"{categoria}_{numero:03d}",
            "categoria": categoria,
            "texto": texto,
        })

frases_base = pd.DataFrame(filas)

ruta = RAIZ / "ml/data/frases_base.csv"

# "x" crea el archivo y evita sobrescribirlo si ya existe.
with ruta.open("x", encoding="utf-8", newline="") as archivo:
    frases_base.to_csv(archivo, index=False)

print("Archivo creado:", ruta)
print("Total de frases base:", len(frases_base))
display(frases_base.groupby("categoria").size().rename("frases_base"))

FileExistsError: [Errno 17] File exists: '/home/saurin/sistema-servicios-ia/ml/data/frases_base.csv'

In [24]:
nuevas_frases = {
    "atasco_papel": [
        "la impresora empieza a tomar la hoja pero se queda detenida a medio camino",
        "cada vez que imprimo por ambas caras el papel queda atrapado dentro",
        "el rodillo gira pero las hojas siguen en la bandeja sin entrar",
    ],
    "bateria": [
        "el porcentaje de carga baja de ochenta a veinte en pocos minutos",
        "la computadora funciona enchufada pero no logra guardar carga",
        "el sistema avisa que debo reemplazar la batería por desgaste",
    ],
    "cabezal_tinta": [
        "en la prueba de inyectores faltan segmentos del color amarillo",
        "después de varios meses sin usarla la impresora de tinta no saca el color negro",
        "la impresión tiene bandas sin color aunque los depósitos están llenos",
    ],
    "disco_duro": [
        "el diagnóstico del almacenamiento muestra una alerta SMART",
        "al copiar archivos aparece un error de lectura y el disco hace golpeteos",
        "el disco aparece y desaparece de la lista de unidades del equipo",
    ],
    "fuente_poder": [
        "la prueba de la fuente de la torre indica que no entrega voltaje",
        "la fuente del gabinete hace un zumbido y el equipo deja de recibir energía",
        "el técnico midió las salidas de la fuente y encontró voltajes fuera de rango",
    ],
    "fusor_laser": [
        "la impresora láser muestra un error de temperatura del fusor",
        "el polvo de tóner queda suelto sobre el papel después de imprimir",
        "la impresora láser no termina de calentar y reporta una falla de la unidad de fijación",
    ],
    "pantalla": [
        "solo se ilumina una mitad de la pantalla del portátil",
        "en un monitor externo la imagen sale bien pero en el panel de la laptop se ve distorsionada",
        "al mover la tapa desaparece la imagen y vuelve cuando la dejo en cierto ángulo",
    ],
    "placa_madre": [
        "el diagnóstico encontró corrosión y un corto en la placa principal",
        "el técnico detectó un componente quemado en la placa madre",
        "la revisión electrónica encontró una falla en el circuito de alimentación de la placa",
    ],
    "sobrecalentamiento": [
        "el procesador supera los cien grados y el equipo reduce mucho su velocidad",
        "el disipador está lleno de polvo y la temperatura sube hasta que se apaga",
        "después de unos minutos de uso aparece una advertencia de temperatura elevada",
    ],
    "software": [
        "una actualización quedó incompleta y windows entra en un ciclo de reparación",
        "el navegador cambia solo la página de inicio y agrega extensiones desconocidas",
        "un programa dejó de abrir después de actualizarlo y muestra un error de instalación",
    ],
    "teclado_touchpad": [
        "la barra espaciadora queda presionada y escribe espacios continuamente",
        "el panel táctil mueve el puntero pero no reconoce los clics",
        "el teclado integrado falla aunque uno conectado por usb funciona correctamente",
    ],
}

In [25]:
originales = pd.read_csv(RAIZ / "ml/data/frases_base.csv")

filas_nuevas = [
    {
        "familia": "",
        "categoria": categoria,
        "texto": texto,
    }
    for categoria, textos in nuevas_frases.items()
    for texto in textos
]

propuestas = pd.DataFrame(filas_nuevas)

ampliadas = pd.concat(
    [originales, propuestas],
    ignore_index=True,
)

ruta_ampliada = RAIZ / "ml/data/frases_base_ampliadas.csv"

with ruta_ampliada.open("x", encoding="utf-8", newline="") as archivo:
    ampliadas.to_csv(archivo, index=False)

print("Frases originales:", len(originales))
print("Frases propuestas:", len(propuestas))
print("Total:", len(ampliadas))

FileExistsError: [Errno 17] File exists: '/home/saurin/sistema-servicios-ia/ml/data/frases_base_ampliadas.csv'

In [26]:
from pathlib import Path
import pandas as pd

RAIZ = Path("/home/saurin/sistema-servicios-ia")

datos = pd.read_csv(
    RAIZ / "ml/data/frases_base_ampliadas.csv",
    keep_default_na=False,
)

# Clasificación inicial que revisaremos a continuación.
datos["origen"] = "cliente"
datos["etapa"] = "recepcion"

# Identifica la procedencia real de estos ejemplos.
datos["tipo_dato"] = "sintetico"

In [27]:
etapas_tecnicas = {
    "el diagnóstico del almacenamiento muestra una alerta SMART":
        "revision",

    "la prueba de la fuente de la torre indica que no entrega voltaje":
        "revision",

    "el técnico midió las salidas de la fuente y encontró voltajes fuera de rango":
        "revision",

    "el diagnóstico encontró corrosión y un corto en la placa principal":
        "diagnostico_confirmado",

    "el técnico detectó un componente quemado en la placa madre":
        "diagnostico_confirmado",

    "la revisión electrónica encontró una falla en el circuito de alimentación de la placa":
        "diagnostico_confirmado",

    "el procesador supera los cien grados y el equipo reduce mucho su velocidad":
        "revision",

    "el disipador está lleno de polvo y la temperatura sube hasta que se apaga":
        "revision",
}

# Comprobamos que los textos coincidan con el archivo.
faltantes = set(etapas_tecnicas) - set(datos["texto"])

if faltantes:
    raise ValueError(
        f"No se encontraron estos textos; revisa si fueron editados: {faltantes}"
    )

es_tecnico = datos["texto"].isin(etapas_tecnicas)

datos.loc[es_tecnico, "origen"] = "tecnico"
datos.loc[es_tecnico, "etapa"] = (
    datos.loc[es_tecnico, "texto"].map(etapas_tecnicas)
)

display(
    datos.groupby(["origen", "etapa"])
    .size()
    .rename("cantidad")
    .reset_index()
)

,origen,etapa,cantidad
0,cliente,recepcion,74
1,tecnico,diagnostico_confirmado,3
2,tecnico,revision,5


In [28]:
ruta_salida = RAIZ / "ml/data/frases_contextualizadas.csv"

with ruta_salida.open("x", encoding="utf-8", newline="") as archivo:
    datos.to_csv(archivo, index=False)

print("Guardado en:", ruta_salida)

FileExistsError: [Errno 17] File exists: '/home/saurin/sistema-servicios-ia/ml/data/frases_contextualizadas.csv'

In [29]:
familias_por_categoria = {
    "bateria": [
        "dependencia_cargador", "autonomia", "carga",
        "hinchazon", "autonomia",
        "autonomia", "carga", "desgaste",
    ],
    "pantalla": [
        "rotura", "defecto_imagen", "sin_imagen",
        "defecto_imagen", "intermitencia",
        "defecto_imagen", "defecto_imagen", "intermitencia",
    ],
    "placa_madre": [
        "dano_electrico", "liquidos", "arranque",
        "arranque", "dano_electrico",
        "liquidos", "dano_electrico", "dano_electrico",
    ],
    "fuente_poder": [
        "sin_energia", "apagados", "dano_electrico",
        "dano_electrico",
        "medicion_voltaje", "apagados", "medicion_voltaje",
    ],
    "disco_duro": [
        "ruido_lectura", "arranque", "arranque",
        "archivos", "lentitud",
        "diagnostico_smart", "ruido_lectura", "desconexion",
    ],
    "software": [
        "lentitud", "malware", "instalacion_sistema",
        "aplicaciones", "error_sistema",
        "error_sistema", "malware", "aplicaciones",
    ],
    "sobrecalentamiento": [
        "temperatura_ruido", "apagado_con_carga",
        "temperatura_ruido", "ventilacion",
        "temperatura_rendimiento", "ventilacion", "temperatura_rendimiento",
    ],
    "teclado_touchpad": [
        "teclas", "liquidos", "touchpad", "teclas",
        "teclas", "touchpad", "teclas",
    ],
    "atasco_papel": [
        "recorrido_papel", "alimentacion_multiple",
        "aviso_atasco", "alimentacion",
        "recorrido_papel", "recorrido_papel", "alimentacion",
    ],
    "cabezal_tinta": [
        "colores_ausentes", "impresion_blanca",
        "manchas", "limpieza_sin_mejora",
        "colores_ausentes", "colores_ausentes", "colores_ausentes",
    ],
    "fusor_laser": [
        "fijacion_toner", "defecto_impresion",
        "ruido_humo", "temperatura_papel",
        "error_temperatura", "fijacion_toner", "error_temperatura",
    ],
}

In [30]:
import runpy

datos = pd.read_csv(
    RAIZ / "ml/data/frases_contextualizadas.csv",
    keep_default_na=False,
)

catalogo = runpy.run_path(
    str(RAIZ / "ml/training/catalogo.py")
)

asignaciones = {}

for categoria, familias in familias_por_categoria.items():
    textos = (
        catalogo["FALLAS"][categoria]["frases"]
        + nuevas_frases[categoria]
    )

    if len(textos) != len(familias):
        raise ValueError(
            f"{categoria}: hay {len(textos)} textos "
            f"pero {len(familias)} familias asignadas"
        )

    for texto, familia in zip(textos, familias):
        asignaciones[(categoria, texto)] = f"{categoria}/{familia}"

claves_archivo = set(zip(datos["categoria"], datos["texto"]))

if claves_archivo != set(asignaciones):
    raise ValueError(
        "Los textos del archivo no coinciden con los ejemplos originales "
        "y las propuestas. Revisa si editaste alguno antes de continuar."
    )

datos["familia"] = [
    asignaciones[(categoria, texto)]
    for categoria, texto in zip(datos["categoria"], datos["texto"])
]

resumen = (
    datos.groupby("categoria")
    .agg(
        frases=("texto", "size"),
        familias=("familia", "nunique"),
    )
)

display(resumen)

,frases,familias
categoria,,
atasco_papel,7,4
bateria,8,5
cabezal_tinta,7,4
disco_duro,8,6
fuente_poder,7,4
fusor_laser,7,5
pantalla,8,4
placa_madre,8,3
sobrecalentamiento,7,4


In [31]:
with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None,
):
    display(
        datos[["familia", "origen", "etapa", "texto"]]
        .sort_values(["familia", "texto"])
    )

ruta_familias = RAIZ / "ml/data/frases_con_familias.csv"

with ruta_familias.open("x", encoding="utf-8", newline="") as archivo:
    datos.to_csv(archivo, index=False)

print("Guardado en:", ruta_familias)

,familia,origen,etapa,texto
51,atasco_papel/alimentacion,cliente,recepcion,el rodillo gira pero las hojas siguen en la bandeja sin entrar
40,atasco_papel/alimentacion,cliente,recepcion,no agarra el papel de la bandeja
38,atasco_papel/alimentacion_multiple,cliente,recepcion,jala varias hojas a la vez
39,atasco_papel/aviso_atasco,cliente,recepcion,dice atasco de papel pero no veo nada atascado
50,atasco_papel/recorrido_papel,cliente,recepcion,cada vez que imprimo por ambas caras el papel queda atrapado dentro
49,atasco_papel/recorrido_papel,cliente,recepcion,la impresora empieza a tomar la hoja pero se queda detenida a medio camino
37,atasco_papel/recorrido_papel,cliente,recepcion,se atasca el papel y sale arrugado
52,bateria/autonomia,cliente,recepcion,el porcentaje de carga baja de ochenta a veinte en pocos minutos
1,bateria/autonomia,cliente,recepcion,"la batería dura muy poco, como 10 minutos"
4,bateria/autonomia,cliente,recepcion,marca 0% de carga y se apaga de golpe


FileExistsError: [Errno 17] File exists: '/home/saurin/sistema-servicios-ia/ml/data/frases_con_familias.csv'

In [32]:
import numpy as np
import pandas as pd

datos = pd.read_csv(
    RAIZ / "ml/data/frases_con_familias.csv",
    keep_default_na=False,
)

# Evaluaremos sugerencias antes del diagnóstico confirmado.
utilizables = datos.loc[
    datos["etapa"].isin(["recepcion", "revision"])
].copy()

print("Frases disponibles:", len(datos))
print("Frases para el experimento:", len(utilizables))
print("Diagnósticos excluidos:", len(datos) - len(utilizables))

assert utilizables["familia"].str.strip().ne("").all(), (
    "Hay frases sin familia asignada"
)

Frases disponibles: 82
Frases para el experimento: 79
Diagnósticos excluidos: 3


In [33]:
generador = np.random.default_rng(42)
familias_validacion = []

for categoria, grupo in utilizables.groupby("categoria", sort=True):
    familias = sorted(grupo["familia"].unique())

    if len(familias) < 2:
        raise ValueError(
            f"{categoria} necesita al menos dos familias para separarse"
        )

    elegida = generador.choice(familias)
    familias_validacion.append(elegida)

es_validacion = utilizables["familia"].isin(familias_validacion)

entrenamiento = utilizables.loc[~es_validacion].copy()
validacion = utilizables.loc[es_validacion].copy()

print("Frases de entrenamiento:", len(entrenamiento))
print("Frases de validación:", len(validacion))

resumen = pd.DataFrame({
    "entrenamiento": entrenamiento.groupby("categoria").size(),
    "validacion": validacion.groupby("categoria").size(),
}).fillna(0).astype(int)

display(resumen)

Frases de entrenamiento: 59
Frases de validación: 20


,entrenamiento,validacion
categoria,,
atasco_papel,5,2
bateria,7,1
cabezal_tinta,6,1
disco_duro,7,1
fuente_poder,5,2
fusor_laser,6,1
pantalla,4,4
placa_madre,4,1
sobrecalentamiento,6,1


In [34]:
familias_train = set(entrenamiento["familia"])
familias_val = set(validacion["familia"])

assert familias_train.isdisjoint(familias_val), (
    "Hay familias compartidas entre entrenamiento y validación"
)

def normalizar(serie):
    return (
        serie.str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

textos_train = set(normalizar(entrenamiento["texto"]))
textos_val = set(normalizar(validacion["texto"]))

assert textos_train.isdisjoint(textos_val), (
    "Hay textos repetidos entre entrenamiento y validación"
)

assert set(entrenamiento["categoria"]) == set(validacion["categoria"]), (
    "Alguna categoría no está representada en ambos grupos"
)

print("Separación verificada: sin familias ni textos exactos compartidos.")

Separación verificada: sin familias ni textos exactos compartidos.


In [35]:
rutas = {
    "entrenamiento": RAIZ / "ml/data/texto_entrenamiento.csv",
    "validacion": RAIZ / "ml/data/texto_validacion.csv",
}

if any(ruta.exists() for ruta in rutas.values()):
    raise FileExistsError(
        "Ya existe alguno de los archivos. No se sobrescribió ninguno."
    )

entrenamiento.to_csv(
    rutas["entrenamiento"], index=False, encoding="utf-8"
)
validacion.to_csv(
    rutas["validacion"], index=False, encoding="utf-8"
)

print("Ambos grupos guardados.")

FileExistsError: Ya existe alguno de los archivos. No se sobrescribió ninguno.

In [36]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf_piloto = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        sublinear_tf=True,
    )),
    ("clasificador", LogisticRegression(max_iter=2000)),
])

tfidf_piloto.fit(
    entrenamiento["texto"],
    entrenamiento["categoria"],
)

pred_tfidf_piloto = tfidf_piloto.predict(validacion["texto"])

print("TF-IDF entrenado y evaluado")

TF-IDF entrenado y evaluado


In [37]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

NOMBRE_MODELO = "dccuchile/bert-base-spanish-wwm-cased"

tokenizador_piloto = AutoTokenizer.from_pretrained(NOMBRE_MODELO)
beto_extractor = AutoModel.from_pretrained(NOMBRE_MODELO)
beto_extractor.eval()

def extraer_vectores_piloto(textos, lote=16):
    textos = list(textos)
    bloques = []

    with torch.inference_mode():
        for inicio in range(0, len(textos), lote):
            entradas = tokenizador_piloto(
                textos[inicio:inicio + lote],
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt",
            )

            tokens = beto_extractor(**entradas).last_hidden_state

            mascara = entradas["attention_mask"].unsqueeze(-1)
            mascara = mascara.to(tokens.dtype)

            vectores = (
                (tokens * mascara).sum(dim=1)
                / mascara.sum(dim=1).clamp(min=1)
            )

            bloques.append(vectores.cpu().numpy())

    return np.concatenate(bloques, axis=0)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 5583.68it/s]
[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [38]:
vectores_train_piloto = extraer_vectores_piloto(
    entrenamiento["texto"]
)

vectores_val_piloto = extraer_vectores_piloto(
    validacion["texto"]
)

beto_piloto = LogisticRegression(max_iter=2000)

beto_piloto.fit(
    vectores_train_piloto,
    entrenamiento["categoria"],
)

pred_beto_piloto = beto_piloto.predict(vectores_val_piloto)

print("Vectores de entrenamiento:", vectores_train_piloto.shape)
print("Vectores de validación:", vectores_val_piloto.shape)

Vectores de entrenamiento: (59, 768)
Vectores de validación: (20, 768)


In [39]:
from sklearn.metrics import accuracy_score, f1_score
from IPython.display import display

y_val = validacion["categoria"].to_numpy()
categorias = sorted(entrenamiento["categoria"].unique())

resultados = []

for nombre, predicciones in [
    ("TF-IDF", pred_tfidf_piloto),
    ("BETO fijo", pred_beto_piloto),
]:
    resultados.append({
        "modelo": nombre,
        "aciertos": int((predicciones == y_val).sum()),
        "total": len(y_val),
        "exactitud_%": 100 * accuracy_score(y_val, predicciones),
        "F1_macro": f1_score(
            y_val,
            predicciones,
            labels=categorias,
            average="macro",
            zero_division=0,
        ),
    })

display(pd.DataFrame(resultados).round(3))

comparacion = validacion[
    ["texto", "categoria", "origen", "familia"]
].copy()

comparacion["pred_tfidf"] = pred_tfidf_piloto
comparacion["pred_beto"] = pred_beto_piloto

hay_error = (
    (comparacion["pred_tfidf"] != comparacion["categoria"])
    | (comparacion["pred_beto"] != comparacion["categoria"])
)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None,
):
    display(comparacion.loc[hay_error])

,modelo,aciertos,total,exactitud_%,F1_macro
0,TF-IDF,2,20,10.0,0.079
1,BETO fijo,8,20,40.0,0.296


,texto,categoria,origen,familia,pred_tfidf,pred_beto
6,se ven lineas de colores en la pantalla,pantalla,cliente,pantalla/defecto_imagen,cabezal_tinta,pantalla
8,tiene una mancha oscura que crece en la pantalla,pantalla,cliente,pantalla/defecto_imagen,disco_duro,fusor_laser
11,se mojo con agua y ya no prende,placa_madre,cliente,placa_madre/liquidos,disco_duro,placa_madre
17,hizo un chispazo y se apago todo,fuente_poder,cliente,fuente_poder/dano_electrico,software,placa_madre
18,huele a plastico quemado cuando la prendo,fuente_poder,cliente,fuente_poder/dano_electrico,bateria,placa_madre
27,no me abre el office ni el navegador,software,cliente,software/aplicaciones,disco_duro,teclado_touchpad
30,se apaga sola cuando juego o abro programas pesados,sobrecalentamiento,cliente,sobrecalentamiento/apagado_con_carga,bateria,bateria
33,varias teclas no responden,teclado_touchpad,cliente,teclado_touchpad/teclas,disco_duro,software
36,algunas letras escriben dos veces,teclado_touchpad,cliente,teclado_touchpad/teclas,disco_duro,cabezal_tinta
44,limpie los cabezales varias veces y sigue igual,cabezal_tinta,cliente,cabezal_tinta/limpieza_sin_mejora,disco_duro,cabezal_tinta


In [42]:
from pathlib import Path
import pandas as pd

RAIZ = Path("/home/saurin/sistema-servicios-ia")
carpeta_resultados = RAIZ / "ml/resultados/experimento_familias"
carpeta_resultados.mkdir(parents=True, exist_ok=True)

# Variables creadas en la celda de comparación del notebook 04.
resumen_resultados = pd.DataFrame(resultados)

resumen_resultados.to_csv(
    carpeta_resultados / "metricas.csv",
    index=False,
    encoding="utf-8",
)

comparacion.to_csv(
    carpeta_resultados / "predicciones_validacion.csv",
    index=False,
    encoding="utf-8",
)

display(resumen_resultados.round(3))
print("Resultados guardados en:", carpeta_resultados)

,modelo,aciertos,total,exactitud_%,F1_macro
0,TF-IDF,2,20,10.0,0.079
1,BETO fijo,8,20,40.0,0.296


Resultados guardados en: /home/saurin/sistema-servicios-ia/ml/resultados/experimento_familias
